In [2]:
from __future__ import annotations

import argparse
import hashlib
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import ollama
from pydantic import BaseModel, Field, ValidationError
from rapidfuzz import fuzz
from tqdm import tqdm


DEFAULT_CHAT_MODEL = "gemma3:12b"
DEFAULT_EMBED_MODEL = "embeddinggemma"
DEFAULT_CACHE_DIR = "./cache_cid_rag"
DEFAULT_OUTPUT_FILE = "resultado_cid_predito.csv"

TOP_K_VECTOR = 12
TOP_K_FINAL_CONTEXT = 8
EMBED_BATCH_SIZE = 64
FUZZY_DIAG_THRESHOLD = 88
FUZZY_TOPO_THRESHOLD = 90
FUZZY_PROC_THRESHOLD = 90


def normalize_text(value: Any) -> str:
    """Normaliza texto para facilitar match e busca."""
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_colname(col: str) -> str:
    text = normalize_text(col)
    replacements = {
        "ç": "c", "ã": "a", "á": "a", "à": "a", "â": "a",
        "é": "e", "ê": "e", "í": "i",
        "ó": "o", "ô": "o", "õ": "o",
        "ú": "u",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def safe_str(value: Any) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def file_md5(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def read_csv_auto(path: Path) -> pd.DataFrame:
    last_error = None
    for enc in ("utf-8-sig", "utf-8", "latin1", "cp1252"):
        for sep in (";", ","):
            try:
                return pd.read_csv(path, dtype=str, encoding=enc, sep=sep)
            except Exception as e:
                last_error = e
    raise RuntimeError(f"Falha ao ler {path}. Último erro: {last_error}")


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def as_list_unique(values: list[str]) -> list[str]:
    out = []
    seen = set()
    for v in values:
        if not v:
            continue
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out


COLUMN_ALIASES = {
    "patient_id": [
        "patient_id", "paciente_id", "id_paciente", "id", "registro",
        "prontuario", "prontuario_bp", "prontuario_bp_", "mrn"
    ],
    "snomed": [
        "snomed", "codigo_snomed", "cod_snomed", "snomed_code",
        "morfologia_snomed", "snomed_morfologia"
    ],
    "cid": [
        "cid", "cid10", "cid_10", "codigo_cid", "cod_cid", "cid_codigo"
    ],
    "diagnostico": [
        "diagnostico", "diagnostico_morfologia", "descricao", "descricao_diagnostico",
        "morfologia", "diag", "termo", "termo_diagnostico"
    ],
    "topografia": [
        "topografia", "topografia_anatomica", "topografia_ana",
        "topografia_anatomica_principal", "sitio_anatomico", "site"
    ],
    "procedimento": [
        "procedimento", "tratamento", "cirurgia", "terapia", "procedimento_realizado"
    ],
    "cr": [
        "cr", "classificacao_cr", "classificacao", "classificacao_final", "grupo_cr"
    ],
}


def find_best_column(df: pd.DataFrame, logical_name: str, required: bool = False) -> str | None:
    aliases = [normalize_colname(x) for x in COLUMN_ALIASES.get(logical_name, [])]
    normalized_map = {normalize_colname(c): c for c in df.columns}

    for alias in aliases:
        if alias in normalized_map:
            return normalized_map[alias]

    for alias in aliases:
        for norm_col, original_col in normalized_map.items():
            if alias in norm_col or norm_col in alias:
                return original_col

    if required:
        raise KeyError(
            f"Não consegui identificar a coluna lógica '{logical_name}'. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None


class CidPrediction(BaseModel):
    predicted_cid: str | None = Field(default=None, description="CID-10 previsto, ex: C50.9")
    confidence: float = Field(ge=0.0, le=1.0, description="Confiança entre 0 e 1")
    needs_review: bool = Field(description="True se o caso precisar de revisão humana")
    rationale: str = Field(description="Resumo curto do porquê")
    source_record_ids: list[str] = Field(default_factory=list, description="IDs dos registros de contexto usados")
    chosen_method: str = Field(description="Método usado: exact_snomed, rag_candidates, rag_only")
    candidate_cids_considered: list[str] = Field(default_factory=list)


@dataclass
class KBRecord:
    record_id: str
    source_name: str
    cid: str
    snomed: str
    diagnostico: str
    topografia: str
    procedimento: str
    cr: str
    text: str
    raw: dict[str, Any]


def build_kb_records(
    snomed_df: pd.DataFrame,
    cr_df: pd.DataFrame,
    snomed_source_name: str,
    cr_source_name: str,
) -> list[KBRecord]:
    records: list[KBRecord] = []

    snomed_col = find_best_column(snomed_df, "snomed", required=True)
    cid_col = find_best_column(snomed_df, "cid", required=True)
    diag_col = find_best_column(snomed_df, "diagnostico", required=False)
    topo_col = find_best_column(snomed_df, "topografia", required=False)
    cr_col_snomed = find_best_column(snomed_df, "cr", required=False)
    cid_desc_col_snomed = None
    for c in snomed_df.columns:
        if normalize_colname(c) == "descricao_cid":
            cid_desc_col_snomed = c
            break

    for i, row in snomed_df.fillna("").iterrows():
        snomed = safe_str(row.get(snomed_col, ""))
        cid = safe_str(row.get(cid_col, ""))
        diag = safe_str(row.get(diag_col, "")) if diag_col else ""
        topo = safe_str(row.get(topo_col, "")) if topo_col else ""
        cid_desc = safe_str(row.get(cid_desc_col_snomed, "")) if cid_desc_col_snomed else ""
        cr_value = safe_str(row.get(cr_col_snomed, "")) if cr_col_snomed else ""

        if not any([snomed, cid, diag, topo, cid_desc, cr_value]):
            continue

        text = (
            f"Fonte: {snomed_source_name}\n"
            f"Tipo: mapa_snomed_cid\n"
            f"Registro_ID: SNOMED_{i}\n"
            f"SNOMED: {snomed}\n"
            f"Diagnostico/Morfologia: {diag}\n"
            f"Topografia: {topo}\n"
            f"CID: {cid}\n"
            f"Descrição CID: {cid_desc}\n"
            f"CR: {cr_value}\n"
        )

        records.append(
            KBRecord(
                record_id=f"SNOMED_{i}",
                source_name=snomed_source_name,
                cid=cid,
                snomed=snomed,
                diagnostico=diag,
                topografia=topo,
                procedimento="",
                cr=cr_value,
                text=text,
                raw=row.to_dict(),
            )
        )

    cid_col_cr = find_best_column(cr_df, "cid", required=True)
    topo_col = find_best_column(cr_df, "topografia", required=False)
    proc_col = find_best_column(cr_df, "procedimento", required=False)
    cr_col = find_best_column(cr_df, "cr", required=False)
    diag_col_cr = find_best_column(cr_df, "diagnostico", required=False)
    snomed_col_cr = find_best_column(cr_df, "snomed", required=False)

    for i, row in cr_df.fillna("").iterrows():
        cid = safe_str(row.get(cid_col_cr, ""))
        topo = safe_str(row.get(topo_col, "")) if topo_col else ""
        proc = safe_str(row.get(proc_col, "")) if proc_col else ""
        cr_value = safe_str(row.get(cr_col, "")) if cr_col else ""
        diag = safe_str(row.get(diag_col_cr, "")) if diag_col_cr else ""
        snomed = safe_str(row.get(snomed_col_cr, "")) if snomed_col_cr else ""

        if not any([cid, topo, proc, cr_value, diag, snomed]):
            continue

        text = (
            f"Fonte: {cr_source_name}\n"
            f"Tipo: base_cid_classificacao_cr\n"
            f"Registro_ID: CR_{i}\n"
            f"CID: {cid}\n"
            f"Diagnostico/Morfologia: {diag}\n"
            f"SNOMED: {snomed}\n"
            f"Topografia: {topo}\n"
            f"Procedimento: {proc}\n"
            f"CR: {cr_value}\n"
        )

        records.append(
            KBRecord(
                record_id=f"CR_{i}",
                source_name=cr_source_name,
                cid=cid,
                snomed=snomed,
                diagnostico=diag,
                topografia=topo,
                procedimento=proc,
                cr=cr_value,
                text=text,
                raw=row.to_dict(),
            )
        )

    if not records:
        raise ValueError("Nenhum registro foi criado para a base de conhecimento.")
    return records


def build_exact_indexes(records: list[KBRecord]) -> dict[str, dict[str, list[KBRecord]]]:
    snomed_index: dict[str, list[KBRecord]] = {}
    cid_index: dict[str, list[KBRecord]] = {}

    for rec in records:
        sn = normalize_text(rec.snomed)
        cid = normalize_text(rec.cid)
        if sn:
            snomed_index.setdefault(sn, []).append(rec)
        if cid:
            cid_index.setdefault(cid, []).append(rec)

    return {
        "snomed": snomed_index,
        "cid": cid_index,
    }


def embed_texts(texts: list[str], model: str, batch_size: int = EMBED_BATCH_SIZE) -> np.ndarray:
    vectors = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Gerando embeddings KB"):
        batch = texts[start:start + batch_size]
        resp = ollama.embed(model=model, input=batch)
        vectors.extend(resp["embeddings"])
    return np.array(vectors, dtype=np.float32)


def prepare_knowledge_base(
    snomed_csv: Path,
    cr_csv: Path,
    embed_model: str,
    cache_dir: Path,
    force_rebuild: bool = False,
) -> tuple[list[KBRecord], np.ndarray, dict[str, dict[str, list[KBRecord]]]]:
    ensure_dir(cache_dir)

    meta_path = cache_dir / "kb_meta.json"
    vectors_path = cache_dir / "kb_vectors.npy"
    records_path = cache_dir / "kb_records.jsonl"

    kb_hash = {
        "snomed_csv": str(snomed_csv),
        "cr_csv": str(cr_csv),
        "snomed_md5": file_md5(snomed_csv),
        "cr_md5": file_md5(cr_csv),
        "embed_model": embed_model,
    }

    if not force_rebuild and meta_path.exists() and vectors_path.exists() and records_path.exists():
        saved = json.loads(meta_path.read_text(encoding="utf-8"))
        if saved == kb_hash:
            records: list[KBRecord] = []
            with open(records_path, "r", encoding="utf-8") as f:
                for line in f:
                    data = json.loads(line)
                    records.append(KBRecord(**data))
            vectors = np.load(vectors_path)
            indexes = build_exact_indexes(records)
            print("Cache da KB reutilizado com sucesso.")
            return records, vectors, indexes

    snomed_df = read_csv_auto(snomed_csv)
    cr_df = read_csv_auto(cr_csv)

    # remover linhas duplicadas / artefatos
    if "SNOMED" in snomed_df.columns:
        snomed_df = snomed_df[snomed_df["SNOMED"].fillna("").str.strip() != ""].copy()

    snomed_df = snomed_df.drop_duplicates()
    cr_df = cr_df.drop_duplicates()

    records = build_kb_records(
        snomed_df=snomed_df,
        cr_df=cr_df,
        snomed_source_name=snomed_csv.name,
        cr_source_name=cr_csv.name,
    )
    texts = [r.text for r in records]
    vectors = embed_texts(texts, model=embed_model)

    np.save(vectors_path, vectors)
    with open(records_path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec.__dict__, ensure_ascii=False) + "\n")
    meta_path.write_text(json.dumps(kb_hash, ensure_ascii=False, indent=2), encoding="utf-8")

    indexes = build_exact_indexes(records)
    return records, vectors, indexes


def cosine_search(query: str, kb_vectors: np.ndarray, embed_model: str, top_k: int = TOP_K_VECTOR) -> list[tuple[int, float]]:
    query_vector = np.array(
        ollama.embed(model=embed_model, input=query)["embeddings"][0],
        dtype=np.float32,
    )
    # As embeddings do Ollama já vêm L2-normalizadas; o produto interno equivale ao cosseno.
    scores = kb_vectors @ query_vector
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]


def fuzzy_candidate_records(
    patient_diag: str,
    patient_topo: str,
    patient_proc: str,
    records: list[KBRecord],
) -> list[tuple[KBRecord, float, str]]:
    candidates: list[tuple[KBRecord, float, str]] = []

    n_diag = normalize_text(patient_diag)
    n_topo = normalize_text(patient_topo)
    n_proc = normalize_text(patient_proc)

    for rec in records:
        best_score = 0.0
        why = []

        if n_diag and rec.diagnostico:
            s = fuzz.token_set_ratio(n_diag, normalize_text(rec.diagnostico))
            if s >= FUZZY_DIAG_THRESHOLD:
                best_score = max(best_score, s / 100.0)
                why.append(f"diag={s}")

        if n_topo and rec.topografia:
            s = fuzz.token_set_ratio(n_topo, normalize_text(rec.topografia))
            if s >= FUZZY_TOPO_THRESHOLD:
                best_score = max(best_score, s / 100.0)
                why.append(f"topo={s}")

        if n_proc and rec.procedimento:
            s = fuzz.token_set_ratio(n_proc, normalize_text(rec.procedimento))
            if s >= FUZZY_PROC_THRESHOLD:
                best_score = max(best_score, s / 100.0)
                why.append(f"proc={s}")

        if best_score > 0:
            candidates.append((rec, best_score, ",".join(why)))

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:TOP_K_VECTOR]


def gather_candidates(
    row: pd.Series,
    records: list[KBRecord],
    kb_vectors: np.ndarray,
    indexes: dict[str, dict[str, list[KBRecord]]],
    embed_model: str,
    patient_columns: dict[str, str | None],
) -> dict[str, Any]:
    patient_snomed = safe_str(row.get(patient_columns.get("snomed", ""), ""))
    patient_diag = safe_str(row.get(patient_columns.get("diagnostico", ""), ""))
    patient_topo = safe_str(row.get(patient_columns.get("topografia", ""), ""))
    patient_proc = safe_str(row.get(patient_columns.get("procedimento", ""), ""))

    exact_records = indexes["snomed"].get(normalize_text(patient_snomed), []) if patient_snomed else []

    query = (
        f"Classificar CID do paciente.\n"
        f"Diagnóstico/Morfologia: {patient_diag}\n"
        f"SNOMED: {patient_snomed}\n"
        f"Topografia: {patient_topo}\n"
        f"Procedimento: {patient_proc}"
    )
    vector_hits = cosine_search(query, kb_vectors, embed_model=embed_model, top_k=TOP_K_VECTOR)
    vector_records = [(records[i], score) for i, score in vector_hits]

    fuzzy_hits = fuzzy_candidate_records(patient_diag, patient_topo, patient_proc, records)

    merged_records: dict[str, dict[str, Any]] = {}

    for rec in exact_records:
        merged_records[rec.record_id] = {
            "record": rec,
            "score": 1.0,
            "evidence": ["exact_snomed"],
        }

    for rec, score in vector_records:
        merged_records.setdefault(
            rec.record_id,
            {"record": rec, "score": score, "evidence": []},
        )
        merged_records[rec.record_id]["score"] = max(merged_records[rec.record_id]["score"], score)
        merged_records[rec.record_id]["evidence"].append("vector")

    for rec, score, why in fuzzy_hits:
        merged_records.setdefault(
            rec.record_id,
            {"record": rec, "score": score, "evidence": []},
        )
        merged_records[rec.record_id]["score"] = max(merged_records[rec.record_id]["score"], score)
        merged_records[rec.record_id]["evidence"].append(f"fuzzy:{why}")

    ranked = sorted(
        merged_records.values(),
        key=lambda x: x["score"],
        reverse=True,
    )[:TOP_K_FINAL_CONTEXT]

    candidate_cids = as_list_unique([safe_str(item["record"].cid) for item in ranked if safe_str(item["record"].cid)])
    exact_cids = as_list_unique([safe_str(rec.cid) for rec in exact_records if safe_str(rec.cid)])

    return {
        "patient_snomed": patient_snomed,
        "patient_diag": patient_diag,
        "patient_topo": patient_topo,
        "patient_proc": patient_proc,
        "exact_records": exact_records,
        "exact_cids": exact_cids,
        "ranked_context": ranked,
        "candidate_cids": candidate_cids,
        "query": query,
    }


def build_llm_messages(patient_case: dict[str, str], candidate_cids: list[str], ranked_context: list[dict[str, Any]]) -> list[dict[str, str]]:
    schema_hint = json.dumps(CidPrediction.model_json_schema(), ensure_ascii=False, indent=2)

    context_lines = []
    for item in ranked_context:
        rec: KBRecord = item["record"]
        context_lines.append(
            json.dumps(
                {
                    "record_id": rec.record_id,
                    "score": round(float(item["score"]), 4),
                    "evidence": item["evidence"],
                    "source_name": rec.source_name,
                    "cid": rec.cid,
                    "snomed": rec.snomed,
                    "diagnostico": rec.diagnostico,
                    "topografia": rec.topografia,
                    "procedimento": rec.procedimento,
                    "cr": rec.cr,
                },
                ensure_ascii=False,
            )
        )

    system_msg = (
        "Você é um classificador de CID-10 para casos oncológicos/clínicos. "
        "Seu trabalho é escolher o CID mais provável usando apenas as evidências fornecidas. "
        "Regras obrigatórias:\n"
        "1) Priorize match exato de SNOMED quando existir.\n"
        "2) Use diagnóstico/morfologia, topografia anatômica e procedimento apenas para desambiguar.\n"
        "3) Não invente CID fora dos candidatos quando houver lista de candidatos.\n"
        "4) Se a evidência estiver fraca ou conflitante, mantenha needs_review=true.\n"
        "5) Retorne somente JSON válido no schema solicitado.\n"
        "6) chosen_method deve ser um de: exact_snomed, rag_candidates, rag_only.\n"
    )

    user_msg = (
        f"Schema JSON obrigatório:\n{schema_hint}\n\n"
        f"CASO DO PACIENTE:\n"
        f"{json.dumps(patient_case, ensure_ascii=False, indent=2)}\n\n"
        f"CANDIDATOS DE CID:\n{json.dumps(candidate_cids, ensure_ascii=False)}\n\n"
        f"CONTEXTO RECUPERADO:\n" + "\n".join(context_lines) + "\n\n"
        "Escolha o CID mais provável. "
        "Se houver candidato fortemente suportado, retorne predicted_cid com confidence coerente. "
        "Se o caso estiver ambíguo, ainda pode retornar o melhor candidato, mas needs_review deve ser true."
    )

    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]


def predict_with_llm(
    chat_model: str,
    patient_case: dict[str, str],
    candidate_cids: list[str],
    ranked_context: list[dict[str, Any]],
) -> CidPrediction:
    messages = build_llm_messages(patient_case, candidate_cids, ranked_context)
    response = ollama.chat(
        model=chat_model,
        messages=messages,
        format=CidPrediction.model_json_schema(),
        options={"temperature": 0},
    )
    content = response["message"]["content"]
    try:
        return CidPrediction.model_validate_json(content)
    except ValidationError as e:
        raise RuntimeError(f"JSON inválido retornado pelo modelo: {content}\nErro: {e}") from e


def classify_row(
    row: pd.Series,
    records: list[KBRecord],
    kb_vectors: np.ndarray,
    indexes: dict[str, dict[str, list[KBRecord]]],
    patient_columns: dict[str, str | None],
    chat_model: str,
    embed_model: str,
) -> dict[str, Any]:
    candidate_info = gather_candidates(
        row=row,
        records=records,
        kb_vectors=kb_vectors,
        indexes=indexes,
        embed_model=embed_model,
        patient_columns=patient_columns,
    )

    patient_case = {
        "diagnostico": candidate_info["patient_diag"],
        "snomed": candidate_info["patient_snomed"],
        "topografia": candidate_info["patient_topo"],
        "procedimento": candidate_info["patient_proc"],
    }

    if len(candidate_info["exact_cids"]) == 1:
        exact_cid = candidate_info["exact_cids"][0]
        exact_ids = [rec.record_id for rec in candidate_info["exact_records"]]
        return {
            "CID_PREDITO": exact_cid,
            "CONFIANCA": 0.99,
            "NEEDS_REVIEW": False,
            "METODO": "exact_snomed",
            "JUSTIFICATIVA": "CID definido por correspondência exata de SNOMED na base de conhecimento.",
            "CANDIDATOS_CID": json.dumps(candidate_info["candidate_cids"], ensure_ascii=False),
            "FONTES_RAG": json.dumps(exact_ids, ensure_ascii=False),
        }

    if not candidate_info["candidate_cids"]:
        return {
            "CID_PREDITO": None,
            "CONFIANCA": 0.0,
            "NEEDS_REVIEW": True,
            "METODO": "no_candidate_found",
            "JUSTIFICATIVA": "Nenhum candidato de CID foi encontrado nas bases após recuperação híbrida.",
            "CANDIDATOS_CID": "[]",
            "FONTES_RAG": "[]",
        }

    pred = predict_with_llm(
        chat_model=chat_model,
        patient_case=patient_case,
        candidate_cids=candidate_info["candidate_cids"],
        ranked_context=candidate_info["ranked_context"],
    )

    if pred.predicted_cid and pred.predicted_cid not in candidate_info["candidate_cids"]:
        pred.needs_review = True
        pred.rationale = (
            pred.rationale
            + " [ALERTA: o modelo retornou CID fora da lista de candidatos recuperados.]"
        )

    return {
        "CID_PREDITO": pred.predicted_cid,
        "CONFIANCA": pred.confidence,
        "NEEDS_REVIEW": pred.needs_review,
        "METODO": pred.chosen_method,
        "JUSTIFICATIVA": pred.rationale,
        "CANDIDATOS_CID": json.dumps(pred.candidate_cids_considered or candidate_info["candidate_cids"], ensure_ascii=False),
        "FONTES_RAG": json.dumps(pred.source_record_ids, ensure_ascii=False),
    }


def classify_file(
    input_csv: Path,
    snomed_csv: Path,
    cr_csv: Path,
    output_csv: Path,
    chat_model: str,
    embed_model: str,
    cache_dir: Path,
    force_rebuild: bool = False,
) -> pd.DataFrame:
    records, kb_vectors, indexes = prepare_knowledge_base(
        snomed_csv=snomed_csv,
        cr_csv=cr_csv,
        embed_model=embed_model,
        cache_dir=cache_dir,
        force_rebuild=force_rebuild,
    )

    df = read_csv_auto(input_csv)

    patient_columns = {
        "patient_id": find_best_column(df, "patient_id", required=False),
        "diagnostico": find_best_column(df, "diagnostico", required=False),
        "snomed": find_best_column(df, "snomed", required=False),
        "topografia": find_best_column(df, "topografia", required=False),
        "procedimento": find_best_column(df, "procedimento", required=False),
    }

    missing = [k for k in ("diagnostico", "snomed", "topografia", "procedimento") if not patient_columns.get(k)]
    if len(missing) == 4:
        raise KeyError(
            "Não consegui identificar nenhuma das colunas esperadas no arquivo de pacientes. "
            f"Colunas disponíveis: {list(df.columns)}"
        )

    outputs = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Classificando pacientes"):
        result = classify_row(
            row=row,
            records=records,
            kb_vectors=kb_vectors,
            indexes=indexes,
            patient_columns=patient_columns,
            chat_model=chat_model,
            embed_model=embed_model,
        )
        outputs.append(result)

    out_df = pd.concat([df.reset_index(drop=True), pd.DataFrame(outputs)], axis=1)
    out_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    return out_df


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="RAG local para classificar CID usando Gemma 3 + Ollama."
    )
    parser.add_argument("--input_csv", required=True, help="Arquivo de pacientes a classificar.")
    parser.add_argument("--snomed_csv", required=True, help="Arquivo de conhecimento SNOMED->CID.")
    parser.add_argument("--cr_csv", required=True, help="Arquivo Base CID_Classificação CR.")
    parser.add_argument("--output_csv", default=DEFAULT_OUTPUT_FILE, help="Arquivo de saída.")
    parser.add_argument("--chat_model", default=DEFAULT_CHAT_MODEL, help="Modelo de chat no Ollama.")
    parser.add_argument("--embed_model", default=DEFAULT_EMBED_MODEL, help="Modelo de embeddings no Ollama.")
    parser.add_argument("--cache_dir", default=DEFAULT_CACHE_DIR, help="Pasta de cache.")
    parser.add_argument("--force_rebuild", action="store_true", help="Reconstrói embeddings da KB.")
    return parser.parse_args()


def main() -> None:
    args = parse_args()

    output_df = classify_file(
        input_csv=Path(args.input_csv),
        snomed_csv=Path(args.snomed_csv),
        cr_csv=Path(args.cr_csv),
        output_csv=Path(args.output_csv),
        chat_model=args.chat_model,
        embed_model=args.embed_model,
        cache_dir=Path(args.cache_dir),
        force_rebuild=args.force_rebuild,
    )

    print("\nConcluído.")
    print(f"Linhas classificadas: {len(output_df)}")
    print(f"Saída salva em: {args.output_csv}")


if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] --input_csv INPUT_CSV
                             --snomed_csv SNOMED_CSV --cr_csv CR_CSV
                             [--output_csv OUTPUT_CSV]
                             [--chat_model CHAT_MODEL]
                             [--embed_model EMBED_MODEL]
                             [--cache_dir CACHE_DIR] [--force_rebuild]
ipykernel_launcher.py: error: argument --force_rebuild: ignored explicit argument 'c:\\Users\\rafae\\AppData\\Roaming\\jupyter\\runtime\\kernel-v323e5538a04b80072b65e0865c0e39048390b2bc0.json'


SystemExit: 2

c:\Users\rafae\miniconda3\envs\ml\Lib\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
